In [1]:
!uv pip install -q "sagemaker>=2.0.0,<3.0.0" "sagemaker[local]" boto3 pytz

In [2]:
!uv pip freeze | grep -E "sagemaker|boto3|pytz"

boto3==1.42.73
pytz==2026.1.post1
sagemaker==2.257.1
sagemaker-core==1.0.77


In [3]:
import os
import subprocess
import sys

import boto3

In [4]:
!docker build --no-cache -q -t credit-risk-training:latest -f /app/give_me_some_credit/sagemaker/training/Dockerfile /app/give_me_some_credit/sagemaker/training/

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/

sha256:9512b59c6a6adc6a1cf7bfd4c464a798da327c69066f7430ce061144315e7aa7


In [5]:
!aws s3 ls s3://data-lake --recursive

2026-03-22 18:28:29         55 BaselineTraining-1774204105-2757/output/model.tar.gz
2026-03-22 18:28:29        185 BaselineTraining-1774204105-2757/output/output.tar.gz
2026-03-22 18:34:55         55 BaselineTraining-1774204492-6820/output/model.tar.gz
2026-03-22 18:34:55        186 BaselineTraining-1774204492-6820/output/output.tar.gz
2026-03-22 19:20:10        405 BaselineTraining-1774207169-5a1a/output/model.tar.gz
2026-03-22 19:20:10        184 BaselineTraining-1774207169-5a1a/output/output.tar.gz
2026-03-22 19:21:32        837 HyperparameterTuning-1774207210-3546/output/model.tar.gz
2026-03-22 19:21:32        184 HyperparameterTuning-1774207210-3546/output/output.tar.gz
2026-03-22 17:07:29          0 bronze/
2026-03-22 18:25:51    7564965 bronze/credit_risk/kaggle/2026-03-22/cs-training.csv
2026-03-22 18:20:54       5928 credit-risk-training-2026-03-22-18-20-54-082/input/code/preprocess.py
2026-03-22 18:20:54       5797 credit-risk-training-2026-03-22-18-20-54-103/input/code/evalu

In [6]:
def get_latest_ingestion_date(bucket, prefix, s3_endpoint=None):
    s3 = boto3.client("s3", endpoint_url=s3_endpoint)

    # We use delimiter='/' to get "folders" inside CommonPrefixes
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix, Delimiter="/")

    dates = []
    for page in pages:
        for obj in page.get("CommonPrefixes", []):
            # Extract 'YYYY-MM-DD' from 'path/ingestion_date=YYYY-MM-DD/'
            folder_name = obj.get("Prefix").split("=")[-1].strip("/")
            dates.append(folder_name)

    if not dates:
        raise ValueError(f"No partitions found in s3://{bucket}/{prefix}")

    # Sorting ensures '2026-03-22' comes after '2026-03-21'
    return sorted(dates)[-1]


# --- Usage in your script ---
S3_ENDPOINT = "http://localstack:4566"
LATEST_DATE = get_latest_ingestion_date(
    bucket="data-lake", prefix="gold/credit_risk/features/", s3_endpoint=S3_ENDPOINT
)

print(f"Detected Latest Date: {LATEST_DATE}")

Detected Latest Date: 2026-03-22


In [7]:
NETWORK = os.environ["NETWORK"]
result = subprocess.run(
    [
        sys.executable,
        "/app/give_me_some_credit/sagemaker/training/sm_pipeline.py",
        "--mode",
        "local",
        "--ingestion-date",
        LATEST_DATE,
        "--s3-endpoint",
        "http://localstack:4566",
        "--n-trials",
        "3",
        "--auc-threshold",
        "0.85",
        "--network",
        NETWORK,
    ],
    capture_output=False,
    text=True,
)
print(f"\nExit code: {result.returncode}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


INFO:sagemaker_pipeline_give_me_some_credit:Args: {'mode': 'local', 'ingestion_date': '2026-03-22', 's3_endpoint': 'http://localstack:4566', 's3_bucket': 'data-lake', 'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'credit_risk_pipeline', 'n_trials': 3, 'auc_threshold': 0.85, 'training_image': 'credit-risk-training:latest', 'network': 'mlops-lab_mlops-lab-net', 'aws_region': 'us-east-1'}
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:sagemaker_session:Local SageMaker session ready (bucket=data-lake)
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for scheduler via: environment_global.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python

time="2026-03-22T19:25:32Z" level=warning msg="/tmp/tmp57taw51g/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-03-22T19:25:32Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmp57taw51g\".\nSet `external: true` to use an existing network"
 Container szkk880clm-algo-1-4wx3u Creating 
 Container szkk880clm-algo-1-4wx3u Created 
Attaching to szkk880clm-algo-1-4wx3u
 Container szkk880clm-algo-1-4wx3u Starting 
 Container szkk880clm-algo-1-4wx3u Started 
szkk880clm-algo-1-4wx3u  | 2026-03-22 19:25:35,351 - preprocess - INFO - Args: {'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'credit_risk_pipeline', 'random_state': 42}
szkk880clm-algo-1-4wx3u  | 2026-03-22 19:25:35,432 - preprocess - INFO - Loaded 149,390 rows, 18 columns
szkk880clm-algo-1-4wx3u  | 2026-03-22 19:25:35,497 - preprocess - INFO -   train: 104,573 rows | default rate: 6.70%
szkk88

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.
INFO:sagemaker.local.entities:Pipeline step 'BaselineTraining' SUCCEEDED.
INFO:sagemaker.local.entities:Starting pipeline step: 'HyperparameterTuning'
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting training job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:


bk2rlrvym5-algo-1-xovdb  | 2026-03-22 19:26:16,008 - train_step - INFO - [val] AUC=0.8698 KS=0.5802
bk2rlrvym5-algo-1-xovdb  | 2026/03/22 19:26:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
bk2rlrvym5-algo-1-xovdb  | 🏃 View run baseline_catboost at: http://mlflow:5000/#/experiments/1/runs/c1de6aeb9e0b45148201436bcde37c72
bk2rlrvym5-algo-1-xovdb  | 🧪 View experiment at: http://mlflow:5000/#/experiments/1
bk2rlrvym5-algo-1-xovdb  | 2026-03-22 19:26:17,872 - botocore.credentials - INFO - Found credentials in environment variables.
bk2rlrvym5-algo-1-xovdb  | 2026-03-22 19:26:17,947 - train_step - INFO - Uploaded baseline_results.json → s3://data-lake/sagemaker/pipeline/baseline/baseline_results.json
bk2rlrvym5-algo-1-xovdb  | 2026-03-22 19:26:17,947 - train_step - INFO - Baseline summary:
bk2rlrvym5-algo-1-xovdb  | 2026-03-22 19:26:17,947 - train_step - INFO 

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.
INFO:sagemaker.local.entities:Pipeline step 'HyperparameterTuning' SUCCEEDED.
INFO:sagemaker.local.entities:Starting pipeline step: 'Evaluation'
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting processing job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:
    

nejt7lg05k-algo-1-l3g4i exited with code 0
 Compose Stopping Aborting on container exit...
 Container nejt7lg05k-algo-1-l3g4i Stopping 
 Container nejt7lg05k-algo-1-l3g4i Stopped 
time="2026-03-22T19:27:41Z" level=warning msg="/tmp/tmpj3rmezal/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-03-22T19:27:41Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpj3rmezal\".\nSet `external: true` to use an existing network"
 Container ero5lut0t1-algo-1-49m1a Creating 
 Container ero5lut0t1-algo-1-49m1a Created 
Attaching to ero5lut0t1-algo-1-49m1a
 Container ero5lut0t1-algo-1-49m1a Starting 
 Container ero5lut0t1-algo-1-49m1a Started 
ero5lut0t1-algo-1-49m1a  | 2026-03-22 19:27:43,600 - evaluate - INFO - Args: {'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'credit_risk_pipeline', 'auc_threshold': 0.85}
ero5lut0t1-algo-1-49m1a  | 2026-03-22 19:27:43,